In [ ]:
import os
import numpy as np
import soundfile as sf  # pip install pysoundfile

# Pfad zum Ordner mit den WAV-Dateien
folder_path = r"C:\Users\PietK\Documents\Python Scripts\VSC\BA Phononenkristalle\testaudio"

peaks = []
rms_values = []

for file_name in os.listdir(folder_path):
    if file_name.lower().endswith(".wav"):
        file_path = os.path.join(folder_path, file_name)

        # WAV-Datei einlesen
        data, samplerate = sf.read(file_path)

        # Falls Stereo -> Mono
        if data.ndim > 1:
            data = np.mean(data, axis=1)

        # Peak berechnen (maximaler Absolutwert)
        peak = np.max(np.abs(data))
        peak_db = 20 * np.log10(peak) if peak > 0 else -np.inf
        peaks.append(peak_db)

        # Index des größten Peaks
        peak_index = np.argmax(np.abs(data))

        # Fenstergröße (0.5 s)
        window_size = int(0.5 * samplerate)
        forerun = 100

        # Start- und Endindex fürs Fenster
        start = max(0, peak_index - forerun)
        end = min(len(data), peak_index + window_size - forerun)

        # Fensterausschnitt
        window_data = data[start:end]

        # RMS berechnen
        if len(window_data) > 0:
            rms = np.sqrt(np.mean(window_data**2))
            rms_db = 20 * np.log10(rms) if rms > 0 else -np.inf
        else:
            rms_db = -np.inf

        rms_values.append(rms_db)

# Ergebnisse ausgeben
for f, p, r in zip(os.listdir(folder_path), peaks, rms_values):
    if f.lower().endswith(".wav"):
        print(f"File: {f}, Peak: {p:.2f} dBFS, RMS (0.5s um Peak): {r:.2f} dBFS")

# print("Peak-Werte (in dB):")
# print(peaks)

def StandardAbweichung (filename, data, run):
    if len(data) == 0:
        return None
    
    summe = sum(data)
    anzahl = len(data)
    mw = summe / anzahl
    print(f"{run} Mittelwert: {mw}")

    rqs = 0
    for i, name in zip(data, filename):
        fehler = (i - mw)**2
        rqs = rqs + fehler
        if fehler > 0.05 * abs(mw):
            print(f"Achtung {run} {name} hat große Abweichung. Fehler: {fehler}")
    sigma = np.sqrt(1/(anzahl-1) * rqs)
    print(f"{run} hat eine Standardabweichung von {sigma}")
    return sigma

sigma_peak = StandardAbweichung (os.listdir(folder_path), peaks, "Peak")
print("")
sigma_rms = StandardAbweichung (os.listdir(folder_path), rms_values, "RMS")



File: 1.wav, Peak: -6.97 dBFS, RMS (0.5s um Peak): -19.53 dBFS
File: 10.wav, Peak: -4.75 dBFS, RMS (0.5s um Peak): -16.64 dBFS
File: 2.wav, Peak: -4.76 dBFS, RMS (0.5s um Peak): -16.64 dBFS
File: 3.wav, Peak: -4.90 dBFS, RMS (0.5s um Peak): -17.62 dBFS
File: 4.wav, Peak: -4.88 dBFS, RMS (0.5s um Peak): -17.02 dBFS
File: 5.wav, Peak: -4.76 dBFS, RMS (0.5s um Peak): -16.37 dBFS
File: 6.wav, Peak: -4.79 dBFS, RMS (0.5s um Peak): -17.39 dBFS
File: 7.wav, Peak: -4.83 dBFS, RMS (0.5s um Peak): -16.41 dBFS
File: 8.wav, Peak: -4.80 dBFS, RMS (0.5s um Peak): -16.40 dBFS
File: 9.wav, Peak: -4.79 dBFS, RMS (0.5s um Peak): -16.21 dBFS

Peak Mittelwert: -5.022975545291434
Achtung Peak 1.wav hat große Abweichung. Fehler: 3.784258266616846
Peak hat eine Standardabweichung von 0.6853577766630099

RMS Mittelwert: -17.022901552905722
Achtung RMS 1.wav hat große Abweichung. Fehler: 6.309070058044127
RMS hat eine Standardabweichung von 0.9970401797020645
